# Lab 01 - Explore Microsoft Web IQ APIs

## Required permissions

A Web IQ profile API key authorized for Web Search, Browse, News, Images and Videos. No Azure RBAC or tenant administrator role is required to run this notebook.

**Goal:** prototype source-backed research workflows: retrieve fresh public evidence, inspect sources, and choose the right API. Allow 40 minutes.

Web IQ is a web intelligence service, not an LLM. Its Azure resource provides billing; API requests go to `https://api.microsoft.ai/v3`.

**Before running:** set `WEBIQ_API_KEY` in the repository's `.env` and run the kernel from the `python-notebooks` directory. See the root README for Python/kernel installation. Only use public, non-sensitive queries. Calls incur usage charges; avoid repeatedly running all cells.

**Outcomes:** explain passages versus full documents, distinguish crawl dates from publication dates, retain source attribution, handle crawl/retry responses, explore training media, and prepare an evidence-backed research handoff. Web IQ retrieves evidence; it is not a sanctions-screening system or a substitute for legal review.

## 1. Configuration and authentication

Settings are loaded from `..\.env`, then `..\.env.azure`, relative to the `python-notebooks` directory. File values override existing environment variables; `.env.azure` wins if both files define the same setting. Keep the API key in `.env`, not in notebook source or outputs.

This lab uses a **Web IQ profile API key** in `x-apikey`, not an Azure resource key. Production Entra authentication requires an app-only token for `https://api.microsoft.ai/.default` and a client ID bound to the Web IQ profile; a developer's `az login` token is not a substitute.


**FYI - Full-document access (is not part of Web IQ):** Web IQ can also return full-document formats (`text`, `html`, `markdown`) with additional profile permission requested through [Web IQ Support](https://aka.ms/webiq-support-form). Without that permission, those formats return HTTP `403`. The parameter tables below focus on passage-only profiles; see [content formats](https://learn.microsoft.com/web-iq/concepts/content-formats) for full-document limits and defaults.

In [53]:
import json
import os
import time

import requests
from dotenv import load_dotenv

load_dotenv(r'..\.env', override=True)
load_dotenv(r'..\.env.azure', override=True)

api_key = os.environ['WEBIQ_API_KEY']
if not api_key.strip():
    raise ValueError('A Web IQ portal API key is required.')
print('Configuration loaded. The API key will not be displayed.')

Configuration loaded. The API key will not be displayed.


## 2. Search the web for grounding passages

The documented starting point is `maxResults=10`, `contentFormat='passage'`, `maxLength=3000`. A passage is query-relevant evidence, not a generated answer or a full webpage.

### Request schema: `POST /v3/search/web`

| Field | Type / required | Default | Meaning / allowed values / limits |
| --- | --- | --- | --- |
| `query` | string / yes | none | Search text, maximum **1000 characters**; minimum not documented. Supports `site:` and `-site:`. |
| `maxResults` | integer / no | `10` | Maximum results requested: **50**; minimum not documented. Fewer results can be returned. |
| `language` | string / no | `en` | Supported 2-letter ISO 639-1 interface language; not a guarantee of document language. |
| `region` | string / no | `US` | Supported 2-letter country/region code, such as `US` or `ES`. |
| `location` | string / no | none | Optional geographic hint, e.g. `lat:40.75;long:-74.00`; if omitted, location can be inferred from the query. Numeric bounds are not documented here. |
| `contentFormat` | string / no | `passage` | Set `passage` explicitly to extract query-relevant text. |
| `maxLength` | integer / no | `3000` | For this lab, request **1-3000 characters per result**. Larger values do not increase the passage-only profile's 3000-character cap. Not query length or total response size. |
| `safeSearch` | string / no | `strict` | `strict` filters adult, racy, violent, hateful and self-harm content. `off` relaxes those filters; illegal content is still blocked. Does not verify accuracy, legal compliance or source trustworthiness. Not an absolute safety guarantee, especially with `site:` queries. |
| `scopeName` | string / no | none | Selects a reusable set of rules that include or exclude domains, URL paths or individual pages from search results. Create and publish it in the Web IQ portal first; for example, an `OfficialSources` scope could include `ofac.treasury.gov`. Not an OAuth permission scope. Name: **1-128** ASCII letters, digits, `.`, `_`, `-`; no whitespace. Not used below. [Scope setup](https://learn.microsoft.com/web-iq/guides/create-scopes). |
| `failOnMissingScope` | boolean / no | `true` | With a named scope, fail if it cannot be applied. `false` permits unscoped fallback. No effect without `scopeName`. |

[Official Web Search reference](https://learn.microsoft.com/web-iq/reference/web-search); [supported languages and regions](https://learn.microsoft.com/web-iq/reference/supported-languages-regions).

In [54]:
question = 'What official OFAC guidance explains how to review a potential sanctions name match?'
response = requests.post(
    'https://api.microsoft.ai/v3/search/web',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': question, 'maxResults': 10,
        'contentFormat': 'passage', 'maxLength': 3000,
        'language': 'en', 'region': 'US', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
if response.status_code >= 400:
    print('HTTP:', response.status_code, response.reason)
    print(response.text[:8000])
    web = {}
else:
    web = response.json()
    print(json.dumps(web, indent=2, ensure_ascii=False))

{
  "webResults": [
    {
      "title": "Assessing OFAC Name Matches | Office of Foreign Assets Control",
      "url": "https://stg2.ofac.treasury.gov/faqs/topic/1591",
      "content": "Assessing OFAC Name Matches | Office of Foreign Assets Control Breadcrumb Home Frequently Asked Questions FAQ Topic Page Assessing OFAC Name Matches Print this topic Assessing OFAC Name Matches 5. When screening for sanctions, how do I determine if I have a valid match to a name on one of OFAC's lists? When attempting to verify or disqualify potential matches to individuals or entities on OFAC's sanctions lists, organizations should follow their own sanctions compliance policies and procedures for investigating potential matches. OFAC recommends that such procedures include the following steps: Step 1: Determine whether the \"alert\" or \"hit\" matches against one of OFAC's sanctions lists (or otherwise involves an OFAC sanctions program) or matches against a list maintained by other government agenci

### Web Search response schema

The response is a JSON object. `webResults` is a list of matching webpages; each item contains the fields explained below. These are **returned fields**, not parameters you send in the request.

| Field | Type | Meaning / limits / availability |
| --- | --- | --- |
| `webResults` | array of objects | Search results; handle an empty or omitted array. Each result is described below. |
| `querySignals.freshness` | boolean | Ranking signal indicating whether fresh documents should be prioritized; not a date filter. |
| `riskLevel` | string | Content-safety rating of the **query itself**, not the returned results: `low`, `moderate`, `high`, `veryHigh`. Not a legal-risk or sanctions-risk assessment. |
| `scopeApplied` | boolean, optional | Whether the requested named scope was applied; may be absent. Zero results do not mean scope failure. |

#### Fields inside each `webResults` item

For example, `web['webResults'][0]['title']` reads the first result's title. The notation `webResults[]` means each item in that list, not a separate JSON field.

| Field | Type | What it tells you / how to use it |
| --- | --- | --- |
| `title` | string | The webpage's title: a label for the result, not an answer to your question or proof of relevance. |
| `url` | string | The source webpage's address. Use it to open the source, cite it, or pass it to Browse. |
| `content` | string | Query-relevant passages retrieved from that webpage. This is the evidence to read or give to a model, not a generated summary. In this lab it is limited by `maxLength`, up to 3000 characters per result; it can be shorter. |
| `lastUpdatedAt` | ISO 8601 string, optional | When the webpage was last updated, if available. It is not necessarily the original publication date or the effective date of a law or policy. Never invent a date if it is missing. |
| `crawledAt` | ISO 8601 string, optional | When the service last crawled the webpage. Helps assess the age of the retrieved copy; it does not say when the author published or changed it. |
| `language` | string | The webpage's language, such as `en` or `es` (ISO 639-1). This can differ from the interface language requested. |
| `isAdult` | boolean | Whether the result is classified as adult content. `false` is not a guarantee of overall safety, accuracy or suitability. |
| `contentTier` | string | Commercial access category: `standard` for the default offering, `entitled` for content accessed through existing publisher licensing/BYOL arrangements, or `premium` for content subject to additional billing. Not a quality score. |
| `clickUrl` | string, optional | A Bing redirect that opens the source and records a user click when click redirection is enabled. It generally leads to the same content as `url`. |
| `domain` | string, conditional | The destination host, when included in the response. |
| `instrumentationSuffix` | string, optional | Per-result identifier used with the telemetry base URLs below when instrumentation is enabled. Not part of the webpage's content. |

String length bounds are not documented for these metadata fields; `content` is controlled by `maxLength` and profile access. Dates and other metadata can be missing or empty, so inspect what actually came back.

### Common response metadata for search endpoints

Now inspect the response above. These fields can also appear in News, Image and Video Search responses.

| Field | Type | Meaning / availability |
| --- | --- | --- |
| `traceId` | string | Service trace identifier for troubleshooting. |
| `instrumentationClickBase` | string | Optional base URL for click telemetry; present only when instrumentation is enabled and click redirection is not enabled. |
| `instrumentationCitationBase` | string | Optional base URL for recording LLM citation usage when instrumentation is enabled. |

**Instrumentation setup:** follow the [Web IQ instrumentation guide](https://learn.microsoft.com/en-us/web-iq/guides/instrumentation) to enable telemetry and report user clicks and LLM citation usage.

### HTTP status information

HTTP `200` means the request succeeded, not that useful results exist. Common failures: `400` invalid input, `401` authentication, `403` permission/content restrictions, `429` rate limiting, and `500`/`503`/`504` service errors. See [error handling](https://learn.microsoft.com/web-iq/reference/error-handling) and [instrumentation](https://learn.microsoft.com/web-iq/guides/instrumentation).

**Checkpoint:** which OFAC sources would you hand to an attorney, and why? Does `crawledAt` mean publication date? Could the first-ranked source still be outdated? What identifying information is missing before a reviewer can assess whether a name match refers to the same person or entity?

## 3. Localization and deliberate source scoping

Try a separate research task: discover Spanish-language guidance from Spain's data protection authority on generative AI and personal data. This is a localization example, not a translation of the sanctions query. The language parameter controls interface language; it does not guarantee that every returned document is in that language. A `site:` operator narrows recall and should be used only when the task needs it. `safeSearch='strict'` is not an absolute content-safety guarantee, especially with `site:`.

**Request schema:** identical to Web Search in section 2. This request changes `query`, sets `language='es'` and `region='ES'`, and uses `site:aepd.es` inside the query. `maxResults=10`, `maxLength=3000`, `contentFormat='passage'` and `safeSearch='strict'` retain the same meanings and bounds.

In [55]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/web',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'inteligencia artificial generativa proteccion de datos personales site:aepd.es',
        'language': 'es', 'region': 'ES', 'safeSearch': 'strict',
        'maxResults': 10, 'contentFormat': 'passage', 'maxLength': 3000,
    },
    timeout=(10, 90),
)
if response.status_code >= 400:
    print('HTTP:', response.status_code, response.reason)
    print(response.text[:8000])
    localized = {}
else:
    localized = response.json()
    print(json.dumps(localized, indent=2, ensure_ascii=False))

{
  "webResults": [
    {
      "title": "https://www.aepd.es/guias/sumario-recomendaciones-iag-aepd.pdf",
      "url": "https://www.aepd.es/guias/sumario-recomendaciones-iag-aepd.pdf",
      "content": "1 / 4 ENERO 2026 RESUMEN BÁSICO DE OBLIGACIONES Y RECOMENDACIONES PARA LA GESTIÓN DE IAG EN LA AEPD En el marco de la ‘Política General para el Uso de IA Generativa en Procesos Administrativos de la AEPD’ publicada en noviembre de 2025 y del anexo que establece un marco práctico para la aplicación segura, ética y controlada de la IA en la AEPD, en diciembre de 2025, se listan de manera no exhaustiva las principales obligaciones y recomendaciones dirigidas a los órganos de decisión y los equipos a cargo de implantar casos de uso de IA Generativa (IAG) en los procesos y tratamientos de la organización: Gobernanza • Identificar de forma precisa los roles de decisión y ejecución conforme a la política interna de uso de IAG, especialmente los responsables funcionales y los responsables técn

### Localized Search response schema

The response uses the same `webResults` schema and common metadata explained after the first Web Search response. Inspect each result's `language`, title, URL and content; the requested interface language does not guarantee that every document is Spanish.

## 4. Browse: from a discovered source to page content

In the previous sections, we used **Search**, which starts with a question and finds relevant pages (`POST /v3/search/web`). Now we will look at **Browse**, which starts with a URL and retrieves content from that specific page (`POST /v3/browse`). These are two separate API calls: Search finds pages; Browse reads a page you choose.

**`liveCrawl` controls whether Browse can fetch a public page directly from its website when the URL is not in Web IQ's index.**

- **`'none'` (default):** return indexed content only; an unindexed URL returns HTTP 404.
- **`'fallback'` (used below):** return indexed content if available; otherwise, request a live crawl. HTTP 202 means the crawl is still processing.

**When it helps:** your application may already know a public URL, such as a newly published announcement or a less-discoverable documentation page, that Web IQ has not indexed yet. Fallback crawling can retrieve that page for grounding instead of stopping at a missing-index result. It does **not** force a refresh of an already-indexed page or guarantee the latest content; inspect `crawledAt` to assess when the page was last crawled.

**Limits:** crawling depends on publisher restrictions and availability. Browse cannot sign in to a website or accept its login credentials or session cookies; the Web IQ API key authenticates only to Web IQ. For protected content, use an authorized API or connector, such as the SharePoint approaches in labs 07-08. Live crawling does not bypass login pages or paywalls.

The earlier Web Search cell stored its JSON response in `web`. Its `webResults` list contains the pages Search found; `web.get('webResults', [])` reads that list, or returns an empty list if there were no results. Below, we select the first result, put its URL in `browse_url`, and print it so you can see exactly which page Browse will read. You can also set `browse_url` to a public URL you already know, without using Search first. The request uses `contentFormat='passage'` and `query=question` to retrieve relevant excerpts from that page.

The loop makes up to four requests. For a pending crawl it waits until `retryAfter`, documented as a UTC timestamp; it also accepts older duration-style responses such as `60s`. Four requests and the two-minute wait cap are **not API limits**: they are notebook safeguards. Missing source URLs, invalid delays, waits longer than two minutes, and unfinished crawls print an explanatory message and stop instead of raising a traceback or being treated as page content.

### Request schema: `POST /v3/browse`

| Field | Type / required | Default | Meaning / allowed values / limits |
| --- | --- | --- | --- |
| `url` | string / yes | none | Public page to retrieve. URL length bounds not documented. |
| `query` | string / yes for `passage` | none | Nonblank text selecting relevant excerpts; maximum **1000 characters**, independent of `maxLength`. |
| `contentFormat` | string / no | `passage` | Set `passage` explicitly to extract query-relevant text. |
| `maxLength` | integer / no | `3000` | For this lab, request **1-3000 content characters**. Larger values do not increase the passage-only profile's 3000-character cap. |
| `liveCrawl` | string / no | `none` | `none`: index only, `404` if absent. `fallback`: request live crawl if unindexed; may return `202`. Not a forced refresh. |
| `renderDynamicPages` | boolean / no | `false` | Enable JavaScript rendering for dynamic-content screenshot; works only with `liveCrawl='fallback'`. |
| `language` | string / no | `en` | Supported 2-letter ISO 639-1 language. |
| `region` | string / no | `US` | Supported 2-letter country/region code. |

[Official Browse reference](https://learn.microsoft.com/web-iq/reference/browse).

In [56]:
from datetime import datetime, timezone

page = {}
search_results = web.get('webResults', [])
if not search_results or not search_results[0].get('url'):
    print('No source URL to browse. Change the search query and rerun search.')
else:
    first_result = search_results[0]
    browse_url = first_result['url']
    print('Reading page:', browse_url)

    for attempt in range(4):
        response = requests.post(
            'https://api.microsoft.ai/v3/browse',
            headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
            json={
                'url': browse_url, 'query': question, 'contentFormat': 'passage',
                'maxLength': 3000, 'liveCrawl': 'fallback',
            },
            timeout=(10, 90),
        )
        if response.status_code not in (200, 202):
            print('HTTP:', response.status_code, response.reason)
            print(response.text[:8000])
            page = {}
            break
        else:
            page = response.json()
            print(json.dumps(page, indent=2, ensure_ascii=False))
        if response.status_code == 200:
            break
        if attempt == 3:
            print('Browse did not complete after four requests. Retry later.')
            page = {}
            break
        if not page.get('retryAfter'):
            print('Pending crawl omitted retryAfter. Retry later manually.')
            page = {}
            break
        delay = page['retryAfter']
        try:
            if delay.endswith('s'):
                seconds = float(delay.removesuffix('s'))
            else:
                retry_at = datetime.fromisoformat(delay.replace('Z', '+00:00'))
                seconds = max(0, (retry_at - datetime.now(timezone.utc)).total_seconds())
        except (AttributeError, TypeError, ValueError):
            print('Could not interpret retryAfter. Retry later manually.')
            page = {}
            break
        if not 0 <= seconds <= 120:
            print(f'Server requested a {seconds}s wait; retry later manually.')
            page = {}
            break
        print(f'Crawl pending. Waiting {seconds:g}s...')
        time.sleep(seconds)

Reading page: https://stg2.ofac.treasury.gov/faqs/topic/1591
{
  "url": "https://stg2.ofac.treasury.gov/faqs/topic/1591",
  "title": "Assessing OFAC Name Matches | Office of Foreign Assets Control",
  "content": "Assessing OFAC Name Matches | Office of Foreign Assets Control Breadcrumb Home Frequently Asked Questions FAQ Topic Page Assessing OFAC Name Matches Print this topic Assessing OFAC Name Matches 5. When screening for sanctions, how do I determine if I have a valid match to a name on one of OFAC's lists? When attempting to verify or disqualify potential matches to individuals or entities on OFAC's sanctions lists, organizations should follow their own sanctions compliance policies and procedures for investigating potential matches. OFAC recommends that such procedures include the following steps: Step 1: Determine whether the \"alert\" or \"hit\" matches against one of OFAC's sanctions lists (or otherwise involves an OFAC sanctions program) or matches against a list maintained b

### Browse response schema

| Field | Type | Meaning / limits / availability |
| --- | --- | --- |
| `url` | string | Page URL, or URL to retry while a crawl is pending. |
| `title` | string | Page title; present only for successful `200` responses. |
| `content` | string | Selected excerpts, bounded by `maxLength` and access; only for `200`. Nonadjacent excerpts can be separated by `...` and cut across sentences. |
| `lastUpdatedAt`, `crawledAt` | ISO 8601 strings | Update and crawl times when available; omitted for `202`. |
| `isAdult` | boolean | Content classification; omitted for `202`. |
| `traceId` | string | Debugging identifier. |
| `retryAfter` | string | Earliest retry time, documented as an ISO 8601 UTC timestamp such as `2026-10-30T01:01:00Z`; present for pending `202` crawls and some errors. |

Browse returns one page object, not a results array. `202` is pending, not usable page content. [Official Browse reference and additional status codes](https://learn.microsoft.com/web-iq/reference/browse).

## 5. News: publisher attribution and current events

Discover reporting on OFAC sanctions enforcement. Treat news as an alerting input for a research workflow, then verify any claimed change against official notices and effective dates. News is not authoritative transaction clearance.

News results can include `source`, `lastUpdatedAt`, `crawledAt` and a thumbnail, but these fields are optional. Compare the publishers and date coverage against Web Search. News Search has a maximum of 20 results; do not assume it supports the video endpoint's `freshness` filter.

### Request schema: `POST /v3/search/news`

| Field | Type / required | Default | Meaning / allowed values / limits |
| --- | --- | --- | --- |
| `query` | string / yes | none | News search text; maximum **1000 characters**; minimum not documented. |
| `maxResults` | integer / no | `10` | Maximum results: **20**; minimum not documented. This example requests `5`. |
| `language` | string / no | `en` | Supported 2-letter interface language. |
| `region` | string / no | `US` | Supported 2-letter country/region code. |
| `location` | string / no | none | Geographic hint, e.g. `lat:40.75;long:-74.00`; numeric bounds not documented here. |
| `contentFormat` | string / no | `passage` | Set `passage` explicitly to extract query-relevant text. |
| `maxLength` | integer / no | `3000` | For this lab, request **1-3000 characters per result**. Larger values do not increase the passage-only profile's 3000-character cap. |
| `scopeName` | string / no | none | Selects a reusable set of rules that include or exclude domains, URL paths or individual pages from news results. Create and publish it in the Web IQ portal first; for example, a scope can restrict results to selected publishers. Not an OAuth permission scope. Name: **1-128** ASCII letters, digits, `.`, `_`, `-`; no whitespace. [Scope setup](https://learn.microsoft.com/web-iq/guides/create-scopes). |
| `failOnMissingScope` | boolean / no | `true` | Scope-failure behavior as in Web Search; `false` allows unscoped fallback. |

`safeSearch` and `freshness` are not listed in the News request reference; do not copy them from other endpoints.

[Official News Search reference](https://learn.microsoft.com/web-iq/reference/news-search).

In [57]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/news',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'OFAC sanctions enforcement developments',
        'maxResults': 5, 'language': 'en', 'region': 'US',
        'contentFormat': 'passage', 'maxLength': 3000,
    },
    timeout=(10, 90),
)
if response.status_code >= 400:
    print('HTTP:', response.status_code, response.reason)
    print(response.text[:8000])
    news = {}
else:
    news = response.json()
    print(json.dumps(news, indent=2, ensure_ascii=False))

{
  "newsResults": [
    {
      "title": "Weekly Sanctions Update: November 2, 2026 | Steptoe",
      "url": "https://www.steptoe.com/en/news-publications/international-compliance-blog/weekly-sanctions-update-november-2-2026.html",
      "content": "Weekly Sanctions Update: November 2, 2026 | Steptoe International Compliance Blog | November 2, 2026 Weekly Sanctions Update: November 2, 2026 Authors Evan T. Abrams, Meredith Rathbone, Karl Hopkins, Guy Soussan, Alexandra Melia, Wendy Wysong, Ali Burney, Quentin Johnson, Algirdas Semeta, Melissa B. Mahle, Anni Coonan, Thomas Goldstein, Zayna Dembinski, Samuel Bloebaum, Elton Smole, Ian Cameron, Wilson Zhao, Zoey Hong, Kiki Dong, Warren H. Geary, Kuangqi Zhong, Andy Xu Sofia, Jake Fitzpatrick, Carter Spahn, Chris Dantes US Developments US Resolves Extreme Networks Case Over Exports to Blocked Russian Defense Manufacturer On October 27, OFAC announced a $1.53 million settlement with Extreme Networks, Inc. (“Extreme”) to resolve nine apparen

### News response schema

| Field | Type | Meaning / limits / availability |
| --- | --- | --- |
| `newsResults` | array of objects | News articles; result count can be less than requested. |
| `scopeApplied` | boolean, optional | Whether a named scope was applied; may be absent. |
| `newsResults[].title`, `.url` | strings | Article title and source URL (see common response metadata after the first Web Search response). |
| `newsResults[].content` | string | Retrieved passages, bounded by `maxLength` and profile access. |
| `newsResults[].source` | string | Publisher name when available. |
| `newsResults[].language` | string | Article language, ISO 639-1. |
| `newsResults[].lastUpdatedAt`, `.crawledAt` | ISO 8601 strings | Update/crawl timestamps when available; not necessarily publication dates. |
| `newsResults[].thumbnail` | object, optional | Preview image: `url` (string), `width` and `height` (integers in pixels; numeric bounds not documented). |
| `newsResults[].isAdult` | boolean | Content classification. |

Also see the common response metadata after the first Web Search response. Response string lengths are not documented except for `content`. [Official News Search reference](https://learn.microsoft.com/web-iq/reference/news-search).

## 6. Image and video discovery

Find sanctions-screening workflow diagrams and OFAC sanctions-compliance training videos. Inspect how a diagram distinguishes a potential name match from a confirmed match and escalation for review. Check the publisher, date and intended audience; a diagram is not authoritative legal guidance.

These are discovery results, not permission to reuse media. After inspecting the image response JSON, run the preview cell to display the returned thumbnails with their titles and source pages. Previews load from external URLs in your notebook viewer; they do not send your Web IQ API key to the image host. Videos remain metadata-only.

Image filters include aspect ratio and minimum size. Video filters include duration, minimum resolution, and discovery-date freshness. An absent summary or moments array simply means that metadata is unavailable.

### Image request schema: `POST /v3/search/images`

| Field | Type / required | Default | Meaning / allowed values / limits |
| --- | --- | --- | --- |
| `query` | string / yes | none | Image search text; maximum **1000 characters**; minimum not documented. |
| `maxResults` | integer / no | `30` | Maximum **30** results; minimum not documented. This example requests `3`. |
| `language`, `region` | strings / no | `en`, `US` | Supported 2-letter language and country/region codes. |
| `aspectRatio` | string / no | no filter | `square` (roughly 1:1), `wide` (width greater than height), `tall` (height greater than width). |
| `color` | string / no | no filter | `colorOnly` or `monochrome`. |
| `safeSearch` | string / no | `strict` | `strict` filters adult, racy, violent, hateful and self-harm content. `off` relaxes those filters; illegal content is still blocked. Does not verify accuracy, legal compliance or source trustworthiness. Not an absolute safety guarantee. |
| `imageSize` | string / no | no filter | `small` (less than 200x200 px), `medium` (width or height 200-500 px), `large` (width or height at least 500 px), `extraLarge` (greater than 1024x768 px). |
| `watermarkFree` | boolean / no | `false` | Exclude watermarked images; not a reuse license. |
| `minHeight`, `maxHeight` | integers / no | no bounds specified | Custom height bounds in pixels; accepted numeric ranges not documented. |
| `minWidth`, `maxWidth` | integers / no | no bounds specified | Custom width bounds in pixels; accepted numeric ranges not documented. |

Custom pixel constraints take precedence over `imageSize` when they conflict. Images do not use `contentFormat` or `maxLength`.

[Official Image Search reference](https://learn.microsoft.com/web-iq/reference/image-search).

In [58]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/images',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'OFAC sanctions screening process flowchart', 'maxResults': 3,
        'aspectRatio': 'wide', 'imageSize': 'large', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
if response.status_code >= 400:
    print('HTTP:', response.status_code, response.reason)
    print(response.text[:8000])
    images = {}
else:
    images = response.json()
    print(json.dumps(images, indent=2, ensure_ascii=False))

{
  "imageResults": [
    {
      "title": "Removal from the OFAC SDN List: Legal Framework, Challenges, and Strategic Considerations",
      "url": "https://www.frilinglaw.com/images/Frame_3.png",
      "hostPageUrl": "https://www.frilinglaw.com/blog/removal-from-ofac-sdn-list",
      "caption": "A flowchart outlining the legal process for removing someone from the OFAC SDN List. It starts with a file petition with OFAC, followed by an OFAC review and inter-agency consultation. The next step is modification, either updated or pending listing. If modified, it leads to a denial. If not modified, it moves to re consideration with new evidence or changed circumstances. Finally, it goes to a judicial review (Federal Court).",
      "width": 1280,
      "height": 844,
      "thumbnailUrl": "https://ts3.mm.bing.net/th?id=OIP.yMDGSxO0VAakLIANiduV3AHaE4&&pid=15.1",
      "lastUpdatedAt": "2025-10-22T21:45:00Z",
      "isAdult": false,
      "clickUrl": "https://api.microsoft.ai/ck/a?!&&p=ba421

### Image response schema

| Field | Type | Meaning / limits / availability |
| --- | --- | --- |
| `imageResults` | array of objects | Image discovery results, not downloaded image bytes. |
| `imageResults[].title` | string | Image title. |
| `imageResults[].url` | string | The image's source URL. |
| `imageResults[].hostPageUrl` | string | Webpage hosting the image; use for source attribution. |
| `imageResults[].thumbnailUrl` | string | Preview-image URL; may be absent. |
| `imageResults[].caption` | string | Generated image description; not necessarily publisher text. |
| `imageResults[].width`, `.height` | integers | Image dimensions in pixels; numeric response bounds not documented. |
| `imageResults[].lastUpdatedAt` | ISO 8601 string | Last update time when available. |
| `imageResults[].isAdult` | boolean | Content classification. |

Also see the common response metadata after the first Web Search response. Response string lengths are not documented. The following preview cell uses `thumbnailUrl`, falling back to `url`; `width=400` is a notebook display size, not an API filter. [Official Image Search reference](https://learn.microsoft.com/web-iq/reference/image-search).

In [59]:
from IPython.display import Image, display

image_results = images.get('imageResults', [])
if not image_results:
    print('No images returned. Try another image search query.')

for image in image_results:
    print('Title:', image.get('title', '(unavailable)'))
    print('Source page:', image.get('hostPageUrl', '(unavailable)'))
    preview_url = image.get('thumbnailUrl') or image.get('url')
    if preview_url and preview_url.startswith(('https://', 'http://')):
        display(Image(url=preview_url, width=400, embed=False))
    else:
        print('No preview URL available.')

Title: Removal from the OFAC SDN List: Legal Framework, Challenges, and Strategic Considerations
Source page: https://www.frilinglaw.com/blog/removal-from-ofac-sdn-list


Title: Vendor Compliance Screening — Sage Intacct Technical Solutions
Source page: https://www.sageintacctmpp.com/customizable-solutions/vendor-compliance-screening


Title: How to choose AML Sanctions Screening System
Source page: https://fincom.co/choose-aml-sanction-screening-system/


### Video request schema: `POST /v3/search/videos`

| Field | Type / required | Default | Meaning / allowed values / limits |
| --- | --- | --- | --- |
| `query` | string / yes | none | Video search text; maximum **1000 characters**; minimum not documented. |
| `maxResults` | integer / no | `30` | Maximum **30** results; minimum not documented. This example requests `3`. |
| `language`, `region` | strings / no | `en`, `US` | Supported 2-letter language and country/region codes. |
| `enablePlaylist` | boolean / no | `false` | Request playlist results; availability is not guaranteed. |
| `freshness` | string / no | no filter | Discovery-date filter: `week`, `month`, `year`, or ISO 8601 interval (`YYYY-MM-DD/YYYY-MM-DD` or UTC timestamp interval). Use `..` for an open interval end. Not publication date. |
| `embeddable` | array of strings / no | no filter | Allowed member: `player` (embeddable player); array-length bounds not documented. |
| `resolution` | string / no | no filter | **Minimum** resolution: `360p`, `480p`, `720p`, `1080p`; `720p` does not mean exactly 720p. |
| `safeSearch` | string / no | `strict` | `strict` filters adult, racy, violent, hateful and self-harm content. `off` relaxes those filters; illegal content is still blocked. Does not verify accuracy, legal compliance or source trustworthiness. Not an absolute safety guarantee. |
| `duration` | string / no | no filter | `short` (under 5 minutes), `medium` (5-20 minutes), `long` (over 20 minutes). |

Videos do not use `contentFormat` or `maxLength`; the response is metadata, not video bytes.

[Official Video Search reference](https://learn.microsoft.com/web-iq/reference/video-search).

In [60]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/videos',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'OFAC sanctions compliance training', 'maxResults': 3,
        'duration': 'medium', 'resolution': '720p',
        'freshness': 'year', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
if response.status_code >= 400:
    print('HTTP:', response.status_code, response.reason)
    print(response.text[:8000])
    videos = {}
else:
    videos = response.json()
    print(json.dumps(videos, indent=2, ensure_ascii=False))

{
  "videoResults": [
    {
      "title": "Webinar Recording: Unpacking OFAC's Sanctions Compliance Guidance",
      "url": "https://www.youtube.com/watch?v=eXWshqxyMPE",
      "description": "Learn How to Design a Program that Ensures Sanctions Risk is Appropriately Managed",
      "publishedBy": "LexisNexis Risk Solutions",
      "viewCount": 5222,
      "length": "00:57:17",
      "allowHttpsEmbedding": true,
      "allowMobileEmbedding": true,
      "embeddingUrl": "https://www.youtube.com/embed/eXWshqxyMPE",
      "lastUpdatedAt": "2020-02-28T13:21:58Z",
      "moments": [],
      "thumbnailUrl": "https://www.bing.com/th?id=OVP.nkGGSar5EGTtLsN8Lfwv9wHgFo&pid=15.1",
      "width": 1920,
      "height": 1080,
      "isAdult": false,
      "clickUrl": "https://api.microsoft.ai/ck/a?!&&p=d86cbd0a19285d4f103330018cd1749dbd3d35cb29eb5688046c9f0f57593f50JmltdHM9MTc5NDY3NjAyMSZpZ3VpZD0yODFmNzA4Ni02YjRkLTRiZDQtYWZkMy1iYjgyYmJiMjEyYzQmbGdtYXA9UDJOcFpEMUZNalEwT0RsQk1qVkRPRFEwTnprM09EZ3dRelU

### Video response schema

| Field | Type | Meaning / limits / availability |
| --- | --- | --- |
| `videoResults` | array of objects | Video discovery results; fewer than `maxResults` can be returned. |
| `playlists` | array of objects, optional | Playlist `title`, `description` (strings), and `videos` array with each video's `title`, `description`, `url`. Optional even when requested. |
| `videoResults[].title`, `.url` | strings | Video title and source URL (see common response metadata after the first Web Search response). |
| `videoResults[].description` | string | Publisher-provided or LLM-generated description. |
| `videoResults[].publishedBy` | string | Uploader/publisher. |
| `videoResults[].viewCount` | 64-bit integer | Reported view count; numeric bounds not documented. |
| `videoResults[].summary` | string, optional | AI-generated summary; verify against the source. |
| `videoResults[].length` | string | Duration in `HH:MM:SS` format. |
| `videoResults[].allowHttpsEmbedding`, `.allowMobileEmbedding` | booleans | Embedding capability indicators, not permission to reuse content. |
| `videoResults[].embeddingUrl` | string | Embeddable player URL when available. |
| `videoResults[].lastUpdatedAt` | string | Formatted last-update date; do not assume it is publication date or always ISO 8601. |
| `videoResults[].bestMomentIndex` | integer | Index of the best segment when moments are available; bounds not documented. |
| `videoResults[].moments` | array of objects, optional | Segment `title`, `description`, `momentUrl` (strings), `startTime`, `endTime` (strings in `d.hh:mm:ss` format). |
| `videoResults[].thumbnailUrl` | string | Preview URL. |
| `videoResults[].width`, `.height` | integers | Video dimensions in pixels; numeric bounds not documented. |
| `videoResults[].isAdult` | boolean | Content classification. |

Several video metadata fields are optional. Also see the common response metadata after the first Web Search response; response string lengths and moment/playlist counts are not documented. [Official Video Search reference](https://learn.microsoft.com/web-iq/reference/video-search).

## Continue to lab 02

Open `02-foundry-web-iq-agent.ipynb` to let a Foundry agent call this same service through MCP. Notice the distinction: REST gives you explicit retrieval control; the agent adds model-driven tool selection, approvals and synthesis.

Before sharing this notebook, clear all outputs. Returned web content, trace IDs and URLs may still reveal the topics you searched for. Follow the root README cleanup steps after the workshop.

References: [REST quickstart](https://learn.microsoft.com/web-iq/get-started/quickstart), [Web Search](https://learn.microsoft.com/web-iq/reference/web-search), [Browse](https://learn.microsoft.com/web-iq/reference/browse), [News](https://learn.microsoft.com/web-iq/reference/news-search), [Images](https://learn.microsoft.com/web-iq/reference/image-search), [Videos](https://learn.microsoft.com/web-iq/reference/video-search).